# V025: checkpointed pruning and experiment queue

Use this wrapper for new A-validation training experiments after loading the unchanged sequence/training components and pruning_components_v025.ipynb. It binds the frozen policy and implementation hashes into checkpoint provenance, verifies the exact cohort and known model variant, and evaluates validation-only stopping after the underlying trainer saves its optimizer/model/RNG state.

A stopped run returns status **pruned**, completed=False, with its best native validation MAE, epoch/steps/runtime, reference curves, policy and source/data hashes, and resumable checkpoint hashes. The queue advances to the next planned experiment. Incomplete/pruned members cannot be silently removed from frozen ensemble weights. Interrupted runs wait for exact resume; genuine failures require review.

Final all-data fits and different confirmation splits lack the original A holdout and are not pruned by this policy. Existing legacy partial runs resume with their original frozen trainer; this new source binding is for new experiments. Cloning a pruned checkpoint to a fresh stage preserves the original files before an explicitly recorded resume.

In [ ]:
"""Pruning-aware new-experiment workflow; V005/V008 sources remain unchanged."""
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,copy,gc,shutil,traceback
import numpy as np
import torch

class PrunedRunV025(RuntimeError):
    def __init__(self,result):
        super().__init__('pruned_by_frozen_validation_policy');self.result=result

def _workflow_sha_v025(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1048576),b''):h.update(block)
    return h.hexdigest()

def _workflow_code_sha_v025(path):
    b=json.loads(Path(path).read_text())
    return hashlib.sha256('\n'.join(''.join(c['source']) for c in b['cells'] if c['cell_type']=='code').encode()).hexdigest()

def _workflow_json_v025(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp');tmp.write_text(json.dumps(value,indent=2,allow_nan=False)+'\n');tmp.replace(path)

def load_pruning_policy_v025(root,policy_path,expected_sha256,allow_synthetic_policy=False):
    root=Path(root);path=Path(policy_path)
    if _workflow_sha_v025(path)!=expected_sha256:raise ValueError('Frozen pruning policy changed')
    policy=json.loads(path.read_text())
    if policy['schema']!='v025-validation-only-pruning-policy-v1':raise ValueError('Unsupported policy')
    if policy.get('origin')!='reviewed_completed_A_validation_curves' and not (allow_synthetic_policy and policy.get('origin')=='synthetic_preflight_only'):
        raise ValueError('Reviewed validation history required')
    if policy['component_code_sha256']!=_workflow_code_sha_v025(root/'research/pruning_components_v025.ipynb'):
        raise ValueError('Policy decision implementation changed')
    for file,digest in policy.get('reference_file_hashes',{}).items():
        if _workflow_sha_v025(root/file)!=digest:raise ValueError('Reference history changed: '+file)
    return policy

def bind_pruning_sources_v025(root,policy_path,base_sources):
    root=Path(root);policy_path=Path(policy_path);result=dict(base_sources or {})
    additions={'pruning_components_v025.ipynb':_workflow_code_sha_v025(root/'research/pruning_components_v025.ipynb'),
        'training_workflow_v025.ipynb':_workflow_code_sha_v025(root/'research/training_workflow_v025.ipynb'),
        policy_path.name:_workflow_sha_v025(policy_path)}
    for key,value in additions.items():
        if key in result and result[key]!=value:raise ValueError('Existing workflow source pin differs: '+key)
        result[key]=value
    return result

def pruning_cohort_v025(x_train,y_train,u_out_train,x_val,y_val,u_out_val,model_spec,train_cfg,feature_names,device,loss_kind):
    raw,y,out=_training_arrays(x_train,y_train,u_out_train)
    vx,vy,vo=_training_arrays(x_val,y_val,u_out_val)
    cfg=_train_configuration(train_cfg);rates=deterministic_lr_schedule(cfg)
    arrays=dict(x_sha256=_array_sha256(raw),y_sha256=_array_sha256(y),out_sha256=_array_sha256(out),
        val_x_sha256=_array_sha256(vx),val_y_sha256=_array_sha256(vy),val_out_sha256=_array_sha256(vo))
    return dict(array_fingerprints=arrays,model_spec=canonical_model_spec(model_spec),feature_names=None if feature_names is None else list(feature_names),
        fit_breaths=len(raw),validation_breaths=len(vx),scored_validation_points=int((vo==0).sum()),
        train_cfg=cfg,budget_epochs=cfg['max_epochs'],planned_LR_schedule=rates,
        objective=dict(loss_kind=loss_kind,curve_aux_weight=.1 if loss_kind=='curve_l1' else 0.),
        numeric_runtime=_v8_runtime(_sequence_device(device)))

def verify_prunable_checkpoint_v025(path,record,budget):
    try:
        saved=torch.load(path,map_location='cpu',weights_only=False)
        best=saved.get('best_payload')
        valid=(saved.get('format_version')==8 and saved['epoch']==record['epoch']==len(saved['epochs'])
            and saved['fingerprint']['mode']=='validation' and not saved['completed']
            and saved['epoch']<budget and saved['config']['max_epochs']==budget
            and isinstance(saved['optimizer_state'],dict) and isinstance(saved['rng_state'],dict)
            and best is not None and best['best_val_mae']==record['best_val_mae']
            and saved['optimizer_steps']==record['cumulative_optimizer_steps'])
        if not valid:return None,'checkpoint_progress_or_resume_state_not_verified'
        return dict(epoch=saved['epoch'],best_epoch=best['best_epoch'],
            best_validation_inspiratory_MAE=best['best_val_mae'],optimizer_steps=saved['optimizer_steps'],
            fingerprint_sha256=pruning_digest_v025(_v8_jsonable(saved['fingerprint']))),'verified'
    except Exception as exc:return None,'checkpoint_not_verified:'+type(exc).__name__

def fit_validate_prunable_v025(x_train,y_train,u_out_train,x_val,y_val,u_out_val,
        model_spec,train_cfg,seed,checkpoint_path,*,project_root,policy_path,expected_policy_sha256,
        variant,run_label,pruning_mode='enabled',mode_reason=None,allow_synthetic_policy=False,**fit_kwargs):
    """New validation experiments only; checkpoint-before-callback is inherited from V008."""
    if pruning_mode not in ('enabled','observe_only'):raise ValueError('Unknown pruning mode')
    if pruning_mode=='observe_only' and not mode_reason:raise ValueError('Record why pruning is observation-only')
    cfg=_train_configuration(train_cfg)
    if cfg['patience']:raise ValueError('Disable legacy early stopping; the frozen pruning rule owns stopping')
    root=Path(project_root).resolve();checkpoint=Path(checkpoint_path);last=_last_path(checkpoint)
    outcome_path=checkpoint.with_name(checkpoint.stem+'.outcome.json')
    decision_log=checkpoint.with_name(checkpoint.stem+'.pruning.jsonl');checkpoint.parent.mkdir(parents=True,exist_ok=True)
    policy=load_pruning_policy_v025(root,policy_path,expected_policy_sha256,allow_synthetic_policy)
    supplied=dict(fit_kwargs);sources=bind_pruning_sources_v025(root,policy_path,supplied.get('source_fingerprints'))
    supplied['source_fingerprints']=sources
    # The underlying trainer reseeds identically; this also makes runtime metadata explicit before fitting.
    seed_sequence(seed)
    cohort=pruning_cohort_v025(x_train,y_train,u_out_train,x_val,y_val,u_out_val,model_spec,cfg,
        supplied.get('feature_names'),supplied.get('device','cuda'),supplied.get('loss_kind','l1'))
    descriptor=dict(cohort=cohort,variant=variant,run_id=run_label,
        factory_identity=_v8_callable_identity(supplied.get('model_factory')),model_source_hashes=sources)
    identity=dict(cohort=cohort,variant=variant,run_id=run_label,seed=int(seed),factory_identity=descriptor['factory_identity'],
        arm=supplied.get('arm','v008_scalar'),train_diagnostic_breaths=int(supplied.get('train_diagnostic_breaths',2048)),
        sources=sources,policy_sha256=expected_policy_sha256,pruning_mode=pruning_mode,mode_reason=mode_reason)
    identity_sha=pruning_digest_v025(identity)
    if outcome_path.exists():
        previous=json.loads(outcome_path.read_text())
        if previous['identity_sha256']!=identity_sha:raise ValueError('Existing experiment identity changed; use a fresh stage')
        if previous['status'] in ('completed','pruned'):
            if _workflow_sha_v025(checkpoint)!=previous['checkpoint_sha256'] or _workflow_sha_v025(last)!=previous['last_checkpoint_sha256']:
                raise ValueError('Existing completed/pruned checkpoint changed')
            return previous
        if not supplied.get('resume'):raise ValueError('An interrupted run requires explicit exact resume')
        old=outcome_path.with_name(outcome_path.name+'.prior-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
        shutil.copy2(outcome_path,old)
    history=[]
    if supplied.get('resume') and last.exists():
        saved=torch.load(last,map_location='cpu',weights_only=False);history=copy.deepcopy(saved['epochs']);del saved
    def append_event(value):
        with decision_log.open('a') as f:f.write(json.dumps(value,allow_nan=False)+'\n')
    external_callback=supplied.pop('on_epoch',None)
    def on_epoch(record):
        history.append(copy.deepcopy(record))
        decision=evaluate_pruning_v025(history,descriptor,policy)
        if pruning_mode=='observe_only':decision=dict(decision,action='continue',reason='explicit_observation_only:'+mode_reason)
        entry=dict(decision)
        entry.update(time_utc=datetime.now(timezone.utc).isoformat(),policy_sha256=expected_policy_sha256,
            run_label=run_label,seed=int(seed),epoch=int(record['epoch']),optimizer_steps=int(record['cumulative_optimizer_steps']),
            validation_inspiratory_MAE=float(record['validation_inspiratory_mae']))
        if decision['action']=='prune':
            verified,why=verify_prunable_checkpoint_v025(last,record,cfg['max_epochs'])
            if verified is None:
                entry.update(action='continue',reason=why);append_event(entry)
            else:
                result=dict(status='pruned',completed=False,identity=identity,identity_sha256=identity_sha,
                    policy_file=str(policy_path),policy_sha256=expected_policy_sha256,decision=decision,
                    stopped_epoch=record['epoch'],budget_epochs=cfg['max_epochs'],
                    elapsed_seconds=record['elapsed_seconds'],**verified,
                    checkpoint_file=str(checkpoint),checkpoint_sha256=_workflow_sha_v025(checkpoint),
                    last_checkpoint_file=str(last),last_checkpoint_sha256=_workflow_sha_v025(last),
                    source_fingerprints=sources,created_utc=datetime.now(timezone.utc).isoformat(),
                    resumption='Clone these immutable checkpoints into a new stage and resume the exact recipe/runtime; record any observation-only override.')
                append_event(dict(entry,resumable_checkpoint_verified=True));_workflow_json_v025(outcome_path,result)
                raise PrunedRunV025(result)
        else:append_event(entry)
        if external_callback is not None:external_callback(record)
    try:
        report=fit_validate_v008(x_train,y_train,u_out_train,x_val,y_val,u_out_val,
            model_spec,cfg,seed,checkpoint,on_epoch=on_epoch,**supplied)
    except PrunedRunV025 as stopped:
        result=stopped.result
        gc.collect()
        if supplied.get('device','cuda')=='cuda' and torch.cuda.is_available():torch.cuda.empty_cache()
        return result
    status='completed' if report['completed'] and report['executed_epochs']==cfg['max_epochs'] else 'interrupted'
    result=dict(status=status,completed=status=='completed',identity=identity,identity_sha256=identity_sha,
        policy_file=str(policy_path),policy_sha256=expected_policy_sha256,report=report,
        checkpoint_file=str(checkpoint),checkpoint_sha256=_workflow_sha_v025(checkpoint),
        last_checkpoint_file=str(last),last_checkpoint_sha256=_workflow_sha_v025(last),
        source_fingerprints=sources,created_utc=datetime.now(timezone.utc).isoformat())
    _workflow_json_v025(outcome_path,result);return result

def run_training_queue_v025(jobs,*,project_root,policy_path,expected_policy_sha256,state_path,allow_synthetic_policy=False):
    """A pruned experiment advances immediately; an interruption waits for exact resume."""
    results=[];state_path=Path(state_path)
    for job in jobs:
        try:
            result=fit_validate_prunable_v025(*job['fit_args'],project_root=project_root,policy_path=policy_path,
                expected_policy_sha256=expected_policy_sha256,variant=job['variant'],run_label=job['id'],
                pruning_mode=job.get('pruning_mode','enabled'),mode_reason=job.get('mode_reason'),
                allow_synthetic_policy=allow_synthetic_policy,**job['fit_kwargs'])
        except Exception as exc:
            _workflow_json_v025(state_path,dict(status='failed_requires_review',results=results,
                failed_job=job['id'],error=repr(exc),created_utc=datetime.now(timezone.utc).isoformat()))
            raise
        results.append(dict(id=job['id'],**result))
        _workflow_json_v025(state_path,dict(status='in_progress',results=results))
        if result['status']=='interrupted':
            _workflow_json_v025(state_path,dict(status='waiting_exact_resume',results=results));return results
        gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    _workflow_json_v025(state_path,dict(status='finished_queue',results=results))
    return results

def candidate_readiness_v025(weights,outcomes):
    missing=[key for key in weights if key not in outcomes or outcomes[key]['status']!='completed']
    return dict(eligible_for_scoring=not missing,incomplete_or_pruned_members=missing,
        weights=copy.deepcopy(weights),rule='Never drop a pruned member or renormalize frozen weights')

def clone_pruned_state_v025(outcome,new_checkpoint_path):
    if outcome['status']!='pruned':raise ValueError('Expected a pruned outcome')
    destination=Path(new_checkpoint_path);destination.parent.mkdir(parents=True,exist_ok=True)
    for old,new,key in [(Path(outcome['checkpoint_file']),destination,'checkpoint_sha256'),
            (Path(outcome['last_checkpoint_file']),_last_path(destination),'last_checkpoint_sha256')]:
        if new.exists():raise FileExistsError('Preserve destination')
        if _workflow_sha_v025(old)!=outcome[key]:raise ValueError('Pruned checkpoint changed')
        shutil.copy2(old,new)
    _workflow_json_v025(destination.with_name(destination.stem+'.resume-origin.json'),
        dict(original_pruned_identity_sha256=outcome['identity_sha256'],original_policy_sha256=outcome['policy_sha256'],
            original_files_preserved=True,reason_required_for_observation_only_resume=True))
    return destination


def predict_completed_v025(outcome,x,*,model_factory=None,device='cpu',batch_size=512):
    if outcome['status']!='completed':raise ValueError('Pruned or interrupted experiments cannot supply a completed candidate')
    meta=outcome['report']['v008']
    return predict_checkpoint_v008(outcome['checkpoint_file'],x,device=device,batch_size=batch_size,
        model_factory=model_factory,expected_arm=meta['arm'],expected_loss_kind=meta['loss_kind'],
        source_fingerprints=outcome['source_fingerprints'])
